In [1]:
import os
os.environ["JAVA_HOME"] = "/cvmfs/soft.ccr.buffalo.edu/versions/2023.01/easybuild/software/Core/java/11.0.16"
os.environ['TMPDIR'] = '/vscratch/grp-songyao/pnfioric/temp_dir'
import hail as hl

# Initializing Hail 
#hl.init()
#hl.init(master='local[40]', min_block_size=1024, spark_conf={'spark.driver.memory': '75g', 'spark.executor.memory': '75g'})
hl.init(
    master='local[16]',  # Reduced from 40 to prevent JVM thread contention & GC lockups
    default_reference="GRCh38",
    spark_conf={
        'spark.driver.memory': '75g',         # Leave RAM headroom for OS / off-heap
        'spark.executor.memory': '75g',
        'spark.network.timeout': '1200s',     # Prevents socket drops during heavy pruning
        'spark.executor.heartbeatInterval': '120s',
        'spark.local.dir': '/vscratch/grp-songyao/pnfioric/temp_dir'
    }
)

Loading BokehJS ...

/projects/rpci/songyao/pnfioric/software/hail_python/lib/python3.9/site-packages/hail/context.py:352: UserWarning:

Using hl.init with a default_reference argument is deprecated. To set a default reference genome after initializing hail, call `hl.default_reference` with an argument to set the default reference genome.

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
Running on Apache Spark version 3.5.0
SparkUI available at http://cpn-d02-35.core.ccr.buffalo.edu:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.133-4c60fddb171a
LOGGING: writing to /user/pnfioric/hail-20260921-0817-0.2.133-4c60fddb171a.log


In [2]:
print("Loading datasets...")
# A. Koenig et al. 2024 reference panel
hgdp_base = "/projects/rpci/shared/references/1000G_HGDP_v3_gnomAD/VCF"
# B. Pathways
pathways_base = "/vscratch/grp-songyao/temp_genotypes/sep_files_0.3"
# C. WCHS
wchs_base = "/vscratch/grp-songyao/pnfioric/temp_dir/WCHS_by_chr"
# Output
merged_base = "/vscratch/grp-songyao/pnfioric/temp_dir/WCHS_Pathways_HGDP_by_chr"
os.makedirs(merged_base, exist_ok=True)

Loading datasets...


In [3]:
chrom = 22
hgdp_vcf = f"{hgdp_base}/hgdp_1kg_chr{chrom}.vcf.bgz"
pathways_prefix = f"{pathways_base}/pw_TOPMed_chr_{chrom}"
wchs_prefix = f"{wchs_base}/WCHS_chr{chrom}"
output_mt = f"{merged_base}/merged_chr{chrom}.mt"

In [4]:
mt_hgdp = hl.import_vcf(
    hgdp_vcf,
    reference_genome="GRCh38",
    force_bgz=True,
    min_partitions=2000
)

In [5]:
mt_pathways = hl.import_plink(
    bed=f"{pathways_prefix}.bed",
    bim=f"{pathways_prefix}.bim",
    fam=f"{pathways_prefix}.fam",
    reference_genome="GRCh38"
)

2026-09-18 17:56:23.207 Hail: INFO: Found 4376 samples in fam file.
2026-09-18 17:56:23.208 Hail: INFO: Found 1216357 variants in bim file.


In [6]:
mt_wchs = hl.import_plink(
    bed=f"{wchs_prefix}.bed",
    bim=f"{wchs_prefix}.bim",
    fam=f"{wchs_prefix}.fam",
    reference_genome="GRCh38"
)

2026-09-18 17:56:25.100 Hail: INFO: Found 6692 samples in fam file.
2026-09-18 17:56:25.101 Hail: INFO: Found 1282495 variants in bim file.


In [4]:
# 1. Define schema standardization helper
def clean_mt(mt):
    # Keep only sample ID ('s') in columns and Genotype ('GT') in entries
    mt = mt.select_cols()
    mt = mt.select_entries(mt.GT)
    
    # Restrict to autosomal biallelic SNPs
    mt = mt.filter_rows(mt.locus.in_autosome())
    mt = mt.filter_rows(hl.len(mt.alleles) == 2)
    mt = mt.filter_rows(hl.is_snp(mt.alleles[0], mt.alleles[1]))
    return mt

In [13]:
# 2. Clean all three MatrixTables
mt_hgdp_clean = clean_mt(mt_hgdp)
mt_pathways_clean = clean_mt(mt_pathways)
mt_wchs_clean = clean_mt(mt_wchs)

# 3. Intersect variant keys across all panels
v_hgdp = mt_hgdp_clean.rows().key_by('locus', 'alleles')
v_pathways = mt_pathways_clean.rows().key_by('locus', 'alleles')
v_wchs = mt_wchs_clean.rows().key_by('locus', 'alleles')

shared_variants = v_hgdp.semi_join(v_pathways).semi_join(v_wchs)

# 4. Semi-join rows to shared variants and union sample columns
mt_hgdp_shared = mt_hgdp_clean.semi_join_rows(shared_variants)
mt_pathways_shared = mt_pathways_clean.semi_join_rows(shared_variants)
mt_wchs_shared = mt_wchs_clean.semi_join_rows(shared_variants)

merged_chr = mt_hgdp_shared.union_cols(mt_pathways_shared).union_cols(mt_wchs_shared)

# 5. Repartition and write the merged chromosome MatrixTable to disk
merged_chr = merged_chr.repartition(500)
merged_chr.write(output_mt, overwrite=True)

print(f"Successfully merged Chromosome {chrom} to {output_mt}")

2026-09-18 19:26:23.238 Hail: INFO: Found 4376 samples in fam file.
2026-09-18 19:26:23.239 Hail: INFO: Found 1216357 variants in bim file.
2026-09-18 19:26:25.553 Hail: INFO: Found 6692 samples in fam file.
2026-09-18 19:26:25.553 Hail: INFO: Found 1282495 variants in bim file.
2026-09-18 19:26:28.150 Hail: INFO: Found 4376 samples in fam file.
2026-09-18 19:26:28.150 Hail: INFO: Found 1216357 variants in bim file.
2026-09-18 19:26:30.431 Hail: INFO: Found 4376 samples in fam file.
2026-09-18 19:26:30.431 Hail: INFO: Found 1216357 variants in bim file.
2026-09-18 19:26:33.050 Hail: INFO: Found 6692 samples in fam file.
2026-09-18 19:26:33.050 Hail: INFO: Found 1282495 variants in bim file.
2026-09-18 19:26:35.884 Hail: INFO: Found 6692 samples in fam file.
2026-09-18 19:26:35.884 Hail: INFO: Found 1282495 variants in bim file.
2026-09-18 19:26:38.335 Hail: INFO: Found 4376 samples in fam file.
2026-09-18 19:26:38.336 Hail: INFO: Found 1216357 variants in bim file.
2026-09-18 19:26:40.

Successfully merged Chromosome 9 to /vscratch/grp-songyao/pnfioric/temp_dir/WCHS_Pathways_HGDP_by_chr/merged_chr9.mt


2026-09-18 19:47:39.327 Hail: INFO: wrote matrix table with 715830 rows and 14466 columns in 500 partitions to /vscratch/grp-songyao/pnfioric/temp_dir/WCHS_Pathways_HGDP_by_chr/merged_chr9.mt


In [16]:
print(f"Executing QC and LD pruning on Chromosome {chrom}...")

# 1. Compute variant QC metrics first to generate 'variant_qc' struct
mt_with_qc = hl.variant_qc(merged_chr)

# 2. Filter variants by call rate and minor allele frequency (MAF)
filtered_mt = mt_with_qc.filter_rows(
    (mt_with_qc.variant_qc.call_rate > 0.95) & 
    (mt_with_qc.variant_qc.AF[1] > 0.05) & 
    (mt_with_qc.variant_qc.AF[1] < 0.95)
)

# 3. Repartition to ensure clean, balanced partitions
filtered_mt = filtered_mt.repartition(500) 

# 4. Checkpoint the filtered MatrixTable to disk (chromosome-specific path)
checkpoint_path = f'/vscratch/grp-songyao/pnfioric/temp_dir/merged_chr{chrom}_filtered_preprune.mt'
filtered_mt = filtered_mt.checkpoint(checkpoint_path, overwrite=True)

# 5. Run LD Pruning on the clean MatrixTable
pruned_ht = hl.ld_prune(
    filtered_mt.GT, 
    r2=0.1, 
    bp_window_size=250000
)

# Checkpoint the lightweight pruned variant Table
pruned_ht_path = f'/vscratch/grp-songyao/pnfioric/temp_dir/pruned_variants_chr{chrom}.ht'
pruned_ht = pruned_ht.checkpoint(pruned_ht_path, overwrite=True)

# 6. Filter MatrixTable to pruned variants and write final pruned output
final_mt = filtered_mt.semi_join_rows(pruned_ht)
final_mt_path = f'{merged_base}/merged_pruned_chr{chrom}.mt'
final_mt.write(final_mt_path, overwrite=True)

print(f"Successfully pruned Chromosome {chrom} and saved to {final_mt_path}")

Executing QC and LD pruning on Chromosome 9...


2026-09-18 19:57:48.744 Hail: INFO: Found 4376 samples in fam file.
2026-09-18 19:57:48.745 Hail: INFO: Found 1216357 variants in bim file.
2026-09-18 19:57:51.051 Hail: INFO: Found 6692 samples in fam file.
2026-09-18 19:57:51.052 Hail: INFO: Found 1282495 variants in bim file.
2026-09-18 19:57:53.068 Hail: INFO: Found 4376 samples in fam file.
2026-09-18 19:57:53.068 Hail: INFO: Found 1216357 variants in bim file.
2026-09-18 19:57:54.965 Hail: INFO: Found 4376 samples in fam file.
2026-09-18 19:57:54.965 Hail: INFO: Found 1216357 variants in bim file.
2026-09-18 19:57:57.155 Hail: INFO: Found 6692 samples in fam file.
2026-09-18 19:57:57.155 Hail: INFO: Found 1282495 variants in bim file.
2026-09-18 19:57:59.320 Hail: INFO: Found 6692 samples in fam file.
2026-09-18 19:57:59.320 Hail: INFO: Found 1282495 variants in bim file.
2026-09-18 19:58:01.891 Hail: INFO: Found 4376 samples in fam file.
2026-09-18 19:58:01.891 Hail: INFO: Found 1216357 variants in bim file.
2026-09-18 19:58:04.

Successfully pruned Chromosome 9 and saved to /vscratch/grp-songyao/pnfioric/temp_dir/WCHS_Pathways_HGDP_by_chr/merged_pruned_chr9.mt


2026-09-18 20:25:59.337 Hail: INFO: wrote matrix table with 11832 rows and 14466 columns in 500 partitions to /vscratch/grp-songyao/pnfioric/temp_dir/WCHS_Pathways_HGDP_by_chr/merged_pruned_chr9.mt


In [18]:
plink_output_prefix = f"{merged_base}/merged_pruned_chr{chrom}"

hl.export_plink(
    final_mt, 
    plink_output_prefix,
    ind_id=final_mt.s,       # Uses sample ID 's'
    fam_id=final_mt.s,       # Sets Family ID equal to Individual ID
    pat_id=None,             # Sets paternal ID to missing (0 in .fam)
    mat_id=None,             # Sets maternal ID to missing (0 in .fam)
    is_female=None           # Sets sex to missing (0 in .fam)
)

print(f"Successfully exported PLINK files to: {plink_output_prefix}.bed / .bim / .fam")

2026-09-18 20:29:51.015 Hail: INFO: merging 501 files totalling 40.8M...) / 500]
2026-09-18 20:29:51.185 Hail: INFO: while writing:
    /vscratch/grp-songyao/pnfioric/temp_dir/WCHS_Pathways_HGDP_by_chr/merged_pruned_chr9.bed
  merge time: 168.930ms
2026-09-18 20:29:51.196 Hail: INFO: merging 500 files totalling 466.8K...


Successfully exported PLINK files to: /vscratch/grp-songyao/pnfioric/temp_dir/WCHS_Pathways_HGDP_by_chr/merged_pruned_chr9.bed / .bim / .fam


2026-09-18 20:29:51.251 Hail: INFO: while writing:
    /vscratch/grp-songyao/pnfioric/temp_dir/WCHS_Pathways_HGDP_by_chr/merged_pruned_chr9.bim
  merge time: 54.386ms


In [5]:
def prepare_plink_chr(chrom):
    output_prefix = f"{merged_base}/merged_raw_chr{chrom}"
    if os.path.exists(f"{output_prefix}.bed"):
        return
    
    # 1. Load single chromosome
    mt_hgdp = hl.import_vcf(f"{hgdp_base}/hgdp_1kg_chr{chrom}.vcf.bgz", reference_genome="GRCh38", force_bgz=True, min_partitions=1000)
    mt_pathways = hl.import_plink(bed=f"{pathways_base}/pw_TOPMed_chr_{chrom}.bed", bim=f"{pathways_base}/pw_TOPMed_chr_{chrom}.bim", fam=f"{pathways_base}/pw_TOPMed_chr_{chrom}.fam", reference_genome="GRCh38")
    mt_wchs = hl.import_plink(bed=f"{wchs_base}/WCHS_chr{chrom}.bed", bim=f"{wchs_base}/WCHS_chr{chrom}.bim", fam=f"{wchs_base}/WCHS_chr{chrom}.fam", reference_genome="GRCh38")

    # 2. Clean & Find Shared Variants
    mt_hgdp_c, mt_pw_c, mt_wchs_c = clean_mt(mt_hgdp), clean_mt(mt_pathways), clean_mt(mt_wchs)
    v_shared = mt_hgdp_c.rows().key_by('locus', 'alleles').semi_join(
        mt_pw_c.rows().key_by('locus', 'alleles')
    ).semi_join(
        mt_wchs_c.rows().key_by('locus', 'alleles')
    )

    # 3. Merge & Export
    merged = mt_hgdp_c.semi_join_rows(v_shared).union_cols(
        mt_pw_c.semi_join_rows(v_shared)
    ).union_cols(
        mt_wchs_c.semi_join_rows(v_shared)
    )

    hl.export_plink(
        merged, 
        output_prefix, 
        ind_id=merged.s, 
        fam_id=merged.s, 
        pat_id=None, 
        mat_id=None, 
        is_female=None
    )

for c in range(13, 23):
    prepare_plink_chr(c)

2026-09-21 12:19:48.781 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:19:48.782 Hail: INFO: Found 902326 variants in bim file.
2026-09-21 12:19:50.115 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:19:50.115 Hail: INFO: Found 942314 variants in bim file.
2026-09-21 12:19:52.266 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:19:52.266 Hail: INFO: Found 902326 variants in bim file.
2026-09-21 12:19:53.473 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:19:53.474 Hail: INFO: Found 942314 variants in bim file.
2026-09-21 12:19:55.236 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:19:55.236 Hail: INFO: Found 902326 variants in bim file.
2026-09-21 12:19:56.383 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:19:56.383 Hail: INFO: Found 902326 variants in bim file.
2026-09-21 12:19:57.513 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:19:57.514 Hail: INFO: Found 942314 variants in bim file.
2026-09-21 12:19:59.451 Hai

2026-09-21 12:29:55.823 Hail: INFO: Coerced sorted VCF - no additional import work to do
2026-09-21 12:30:03.204 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:30:03.204 Hail: INFO: Found 818233 variants in bim file.
2026-09-21 12:30:04.255 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:30:04.255 Hail: INFO: Found 859529 variants in bim file.
2026-09-21 12:30:05.248 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:30:05.248 Hail: INFO: Found 818233 variants in bim file.
2026-09-21 12:30:06.247 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:30:06.247 Hail: INFO: Found 818233 variants in bim file.
2026-09-21 12:30:07.292 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:30:07.293 Hail: INFO: Found 859529 variants in bim file.
2026-09-21 12:30:09.105 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:30:09.105 Hail: INFO: Found 859529 variants in bim file.
2026-09-21 12:30:10.165 Hail: INFO: Found 4376 samples in fam file.
2026-09-2

2026-09-21 12:39:43.113 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:39:43.114 Hail: INFO: Found 926682 variants in bim file.
2026-09-21 12:39:44.333 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:39:44.334 Hail: INFO: Found 926682 variants in bim file.
2026-09-21 12:39:45.503 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:39:45.503 Hail: INFO: Found 881994 variants in bim file.
2026-09-21 12:39:46.705 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:39:46.706 Hail: INFO: Found 926682 variants in bim file.
2026-09-21 12:39:55.990 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:39:55.991 Hail: INFO: Found 881994 variants in bim file.
2026-09-21 12:39:57.276 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:39:57.276 Hail: INFO: Found 926682 variants in bim file.
2026-09-21 12:39:58.420 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:39:58.420 Hail: INFO: Found 881994 variants in bim file.
2026-09-21 12:39:59.570 Hai

2026-09-21 12:49:38.664 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:49:38.664 Hail: INFO: Found 824272 variants in bim file.
2026-09-21 12:49:39.726 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:49:39.727 Hail: INFO: Found 782048 variants in bim file.
2026-09-21 12:49:40.782 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:49:40.782 Hail: INFO: Found 782048 variants in bim file.
2026-09-21 12:49:41.857 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:49:41.857 Hail: INFO: Found 824272 variants in bim file.
2026-09-21 12:49:42.919 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:49:42.919 Hail: INFO: Found 824272 variants in bim file.
2026-09-21 12:49:43.890 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 12:49:43.891 Hail: INFO: Found 782048 variants in bim file.
2026-09-21 12:49:44.977 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 12:49:44.977 Hail: INFO: Found 824272 variants in bim file.
2026-09-21 12:57:26.817 Hai

2026-09-21 13:06:36.141 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 13:06:36.141 Hail: INFO: Found 619950 variants in bim file.
2026-09-21 13:06:36.922 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:06:36.922 Hail: INFO: Found 652721 variants in bim file.
2026-09-21 13:06:37.872 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 13:06:37.872 Hail: INFO: Found 619950 variants in bim file.
2026-09-21 13:06:38.630 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 13:06:38.630 Hail: INFO: Found 619950 variants in bim file.
2026-09-21 13:06:39.427 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:06:39.427 Hail: INFO: Found 652721 variants in bim file.
2026-09-21 13:06:40.387 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:06:40.387 Hail: INFO: Found 652721 variants in bim file.
2026-09-21 13:06:41.221 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 13:06:41.221 Hail: INFO: Found 619950 variants in bim file.
2026-09-21 13:06:42.027 Hai

2026-09-21 13:15:20.241 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:15:20.241 Hail: INFO: Found 663622 variants in bim file.
2026-09-21 13:15:21.053 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:15:21.053 Hail: INFO: Found 663622 variants in bim file.
2026-09-21 13:15:21.850 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 13:15:21.850 Hail: INFO: Found 630335 variants in bim file.
2026-09-21 13:15:22.721 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:15:22.721 Hail: INFO: Found 663622 variants in bim file.
2026-09-21 13:15:28.743 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 13:15:28.743 Hail: INFO: Found 630335 variants in bim file.
2026-09-21 13:15:29.676 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:15:29.676 Hail: INFO: Found 663622 variants in bim file.
2026-09-21 13:15:30.468 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 13:15:30.468 Hail: INFO: Found 630335 variants in bim file.
2026-09-21 13:15:31.277 Hai

2026-09-21 13:23:52.821 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:23:52.821 Hail: INFO: Found 402886 variants in bim file.
2026-09-21 13:23:53.350 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 13:23:53.350 Hail: INFO: Found 378645 variants in bim file.
2026-09-21 13:23:53.831 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 13:23:53.831 Hail: INFO: Found 378645 variants in bim file.
2026-09-21 13:23:54.394 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:23:54.394 Hail: INFO: Found 402886 variants in bim file.
2026-09-21 13:23:54.895 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:23:54.896 Hail: INFO: Found 402886 variants in bim file.
2026-09-21 13:23:55.441 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 13:23:55.441 Hail: INFO: Found 378645 variants in bim file.
2026-09-21 13:23:55.942 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:23:55.942 Hail: INFO: Found 402886 variants in bim file.
2026-09-21 13:23:59.988 Hai

2026-09-21 13:31:29.821 Hail: INFO: Found 4376 samples in fam file.
2026-09-21 13:31:29.821 Hail: INFO: Found 384225 variants in bim file.
2026-09-21 13:31:30.370 Hail: INFO: Found 6692 samples in fam file.
2026-09-21 13:31:30.371 Hail: INFO: Found 407232 variants in bim file.
2026-09-21 13:38:09.362 Hail: INFO: merging 1066 files totalling 775.6M... 1065]
2026-09-21 13:38:10.720 Hail: INFO: while writing:
    /vscratch/grp-songyao/pnfioric/temp_dir/WCHS_Pathways_HGDP_by_chr/merged_raw_chr22.bed
  merge time: 1.358s
2026-09-21 13:38:10.765 Hail: INFO: merging 1065 files totalling 9.0M...
2026-09-21 13:38:10.879 Hail: INFO: while writing:
    /vscratch/grp-songyao/pnfioric/temp_dir/WCHS_Pathways_HGDP_by_chr/merged_raw_chr22.bim
  merge time: 114.118ms


In [6]:
import subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_plink_qc_prune(chrom):
    raw_prefix = f"{merged_base}/merged_raw_chr{chrom}"
    prune_temp = f"{merged_base}/temp_prune_chr{chrom}"
    final_prefix = f"{merged_base}/merged_pruned_chr{chrom}"
    
    # 1. QC & Find Independent Pairwise Variants (MAF > 0.05, Call Rate > 0.95, r^2 < 0.1, 250kb window)
    cmd_indep = [
        "plink",
        "--bfile", raw_prefix,
        "--geno", "0.05",
        "--maf", "0.05",
        "--max-maf", "0.95",
        "--snps-only",
        "--indep-pairwise", "250kb", "1", "0.1",
        "--threads", "2",  # 2 threads per chromosome
        "--out", prune_temp
    ]
    subprocess.run(cmd_indep, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # 2. Extract Pruned Variants to Final PLINK Binary Fileset
    cmd_extract = [
        "plink",
        "--bfile", raw_prefix,
        "--extract", f"{prune_temp}.prune.in",
        "--make-bed",
        "--threads", "2",
        "--out", final_prefix
    ]
    subprocess.run(cmd_extract, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # Cleanup intermediate prune files
    for ext in ['.prune.in', '.prune.out', '.log', '.nosex']:
        if os.path.exists(f"{prune_temp}{ext}"):
            os.remove(f"{prune_temp}{ext}")
            
    return f"Chromosome {chrom} finished successfully."

# Execute 11 parallel workers (using 2 threads per worker = 22 threads total)
print("Starting parallel PLINK QC and LD Pruning...")
with ProcessPoolExecutor(max_workers=11) as executor:
    futures = [executor.submit(run_plink_qc_prune, c) for c in range(1, 23)]
    for future in as_completed(futures):
        print(future.result())

print("All 22 chromosomes pruned and saved successfully!")

Starting parallel PLINK QC and LD Pruning...


FileNotFoundError: [Errno 2] No such file or directory: 'plink'

In [ ]:
# Create a merge list file for PLINK
merge_list_path = f"{merged_base}/merge_list.txt"
with open(merge_list_path, "w") as f:
    for c in range(2, 23):
        f.write(f"{merged_base}/merged_pruned_chr{c}\n")

# Combine all chromosome PLINK files into one genome-wide dataset
final_all_chr_cmd = [
    "plink",
    "--bfile", f"{merged_base}/merged_pruned_chr1",
    "--merge-list", merge_list_path,
    "--make-bed",
    "--out", f"{merged_base}/merged_pruned_genome_wide"
]
subprocess.run(final_all_chr_cmd, check=True)
print("Genome-wide merge complete!")

# Step 2: Harmonize and Clean Fields

In [5]:
def clean_mt(mt):
    # Standardize column schema and genotype entries
    mt = mt.select_cols()
    mt = mt.select_entries(mt.GT)
    
    # Filter variants: autosomal, bi-allelic SNPs only
    mt = mt.filter_rows(mt.locus.in_autosome())
    mt = mt.filter_rows(hl.len(mt.alleles) == 2)
    mt = mt.filter_rows(hl.is_snp(mt.alleles[0], mt.alleles[1]))
    
    return mt

# Apply cleaning
ref_mt = clean_mt(mt_hgdp_1kg)
ref_mt = ref_mt.repartition(1000) # Prevents Java 32-bit array overflow
ref_mt = ref_mt.checkpoint('/vscratch/grp-songyao/pnfioric/temp_dir/ref_hgdp_clean.mt', overwrite=True)

dataset2_mt = clean_mt(dataset2_mt)
dataset2_mt = dataset2_mt.repartition(500)
dataset2_mt = dataset2_mt.checkpoint('/vscratch/grp-songyao/pnfioric/temp_dir/dataset2_clean.mt', overwrite=True)

#dataset3_mt = clean_mt(dataset3_mt)
print("Processing Dataset 3 (Pathways)...")
dataset3_mt = clean_mt(dataset3_mt)
dataset3_mt = dataset3_mt.repartition(1000)
dataset3_mt = dataset3_mt.checkpoint('/vscratch/grp-songyao/pnfioric/temp_dir/dataset3_clean.mt', overwrite=True)

2026-09-15 14:23:12.480 Hail: INFO: scanning VCF for sortedness...
2026-09-15 14:25:07.097 Hail: INFO: Coerced sorted VCF - no additional import work to do
2026-09-15 15:12:26.530 Hail: INFO: wrote matrix table with 67089424 rows and 3398 columns in 1000 partitions to /vscratch/grp-songyao/pnfioric/temp_dir/ref_hgdp_clean.mt
2026-09-15 15:13:06.664 Hail: INFO: Found 6692 samples in fam file.
2026-09-15 15:13:06.664 Hail: INFO: Found 29399410 variants in bim file.


FatalError: NegativeArraySizeException: -2147483645

Java stack trace:
com.esotericsoftware.kryo.KryoException: java.lang.NegativeArraySizeException: -2147483645
Serialization trace:
values (org.apache.spark.sql.catalyst.expressions.GenericRow)
locusAlleles (is.hail.io.plink.PlinkVariant)
	at com.esotericsoftware.kryo.serializers.ObjectField.write(ObjectField.java:101)
	at com.esotericsoftware.kryo.serializers.FieldSerializer.write(FieldSerializer.java:508)
	at com.esotericsoftware.kryo.Kryo.writeObject(Kryo.java:575)
	at com.esotericsoftware.kryo.serializers.ObjectField.write(ObjectField.java:79)
	at com.esotericsoftware.kryo.serializers.FieldSerializer.write(FieldSerializer.java:508)
	at com.esotericsoftware.kryo.Kryo.writeClassAndObject(Kryo.java:651)
	at com.esotericsoftware.kryo.serializers.DefaultArraySerializers$ObjectArraySerializer.write(DefaultArraySerializers.java:361)
	at com.esotericsoftware.kryo.serializers.DefaultArraySerializers$ObjectArraySerializer.write(DefaultArrayS

# Step 3: Intersect Shared Variants


In [7]:
print("Finding shared variants across all 3 datasets...")

# Intersect loci and alleles across datasets
v_ref = ref_mt.rows().key_by('locus', 'alleles')
v_ds2 = dataset2_mt.rows().key_by('locus', 'alleles')
v_ds3 = dataset3_mt.rows().key_by('locus', 'alleles')

shared_variants = v_ref.semi_join(v_ds2).semi_join(v_ds3)

# Checkpoint the shared variant key table (lightweight operation)
shared_variants = shared_variants.checkpoint(
    '/vscratch/grp-songyao/pnfioric/temp_dir/shared_variants.ht', 
    overwrite=True
)

# Filter each MT to only shared variants
ref_mt = ref_mt.semi_join_rows(shared_variants)
dataset2_mt = dataset2_mt.semi_join_rows(shared_variants)
dataset3_mt = dataset3_mt.semi_join_rows(shared_variants)

Finding shared variants across all 3 datasets...


NameError: name 'ref_mt' is not defined

# Step 4: Union Samples (Merge Matrix Tables)

In [6]:
print("Merging sample columns across datasets...")

# Union sample columns
merged_mt = ref_mt.union_cols(dataset2_mt)
merged_mt = merged_mt.union_cols(dataset3_mt)

# CRITICAL CHECKPOINT: Sever DAG completely before any QC or LD Pruning
merged_mt = merged_mt.checkpoint(
    '/vscratch/grp-songyao/pnfioric/temp_dir/merged_common_variants.mt', 
    overwrite=True
)

print("Merging complete. MatrixTable is safely checkpointed to disk.")

Merging sample columns across datasets...


FatalError: ArrayIndexOutOfBoundsException: Index 1510637049 out of bounds for length 1073741866

Java stack trace:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 3.0 failed 1 times, most recent failure: Lost task 0.0 in stage 3.0 (TID 192) (cpn-c01-07.core.ccr.buffalo.edu executor driver): java.lang.ArrayIndexOutOfBoundsException: Index 1510637049 out of bounds for length 1073741866
	at com.esotericsoftware.kryo.util.IdentityObjectIntMap.get(IdentityObjectIntMap.java:322)
	at com.esotericsoftware.kryo.util.MapReferenceResolver.getWrittenId(MapReferenceResolver.java:46)
	at com.esotericsoftware.kryo.Kryo.writeReferenceOrNull(Kryo.java:671)
	at com.esotericsoftware.kryo.Kryo.writeClassAndObject(Kryo.java:646)
	at org.apache.spark.serializer.KryoSerializationStream.writeObject(KryoSerializer.scala:278)
	at org.apache.spark.serializer.SerializerHelper$.serializeToChunkedBuffer(SerializerHelper.scala:42)
	at org.apache.spark.executor.Executor$TaskRunner.

# Step 5: Quality Control & LD Pruning

In [8]:
print("Executing QC and LD pruning...")

# 1. Apply basic variant filters (QC, MAF, call rate)
filtered_mt = merged_mt.filter_rows(
    (merged_mt.variant_qc.call_rate > 0.95) & 
    (merged_mt.variant_qc.AF[1] > 0.05) & 
    (merged_mt.variant_qc.AF[1] < 0.95)
)

# 2. Repartition to smaller chunks to reduce per-partition memory footprint
# Target roughly 1,000–5,000 variants per partition
filtered_mt = filtered_mt.repartition(500) 

# 3. CRITICAL: Checkpoint to disk to sever the Spark DAG lineage
checkpoint_path = '/vscratch/grp-songyao/pnfioric/temp_dir/merged_filtered_preprune.mt'
filtered_mt = filtered_mt.checkpoint(checkpoint_path, overwrite=True)

# 4. Run LD Pruning on the clean, checkpointed MatrixTable
# Using memory_per_core or smaller bp_window_size if working on dense data
pruned_ht = hl.ld_prune(
    filtered_mt.GT, 
    r2=0.1, 
    bp_window_size=250000 # 250kb window
)

# 5. Filter MatrixTable to pruned variants and write output
final_mt = filtered_mt.semi_join_rows(pruned_ht)
final_mt.write('/vscratch/grp-songyao/pnfioric/temp_dir/merged_pruned.mt', overwrite=True)

Executing QC and LD pruning...


2026-08-29 22:48:01.295 Hail: INFO: Found 6692 samples in fam file.
2026-08-29 22:48:01.296 Hail: INFO: Found 29399410 variants in bim file.
2026-08-29 22:48:04.856 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:48:04.856 Hail: INFO: Found 2220522 variants in bim file.
2026-08-29 22:48:08.682 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:48:08.682 Hail: INFO: Found 2389049 variants in bim file.
2026-08-29 22:48:11.801 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:48:11.801 Hail: INFO: Found 2012050 variants in bim file.
2026-08-29 22:48:14.911 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:48:14.911 Hail: INFO: Found 2022147 variants in bim file.
2026-08-29 22:48:17.677 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:48:17.677 Hail: INFO: Found 1844283 variants in bim file.
2026-08-29 22:48:20.583 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:48:20.583 Hail: INFO: Found 1792208 variants in bim file.
2026-08-29 22:48:23

2026-08-29 22:51:43.326 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:51:43.327 Hail: INFO: Found 1015974 variants in bim file.
2026-08-29 22:51:44.878 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:51:44.878 Hail: INFO: Found 902326 variants in bim file.
2026-08-29 22:51:51.694 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:51:51.694 Hail: INFO: Found 818233 variants in bim file.
2026-08-29 22:51:53.124 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:51:53.125 Hail: INFO: Found 881994 variants in bim file.
2026-08-29 22:51:54.412 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:51:54.412 Hail: INFO: Found 782048 variants in bim file.
2026-08-29 22:51:55.798 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:51:55.798 Hail: INFO: Found 799425 variants in bim file.
2026-08-29 22:51:56.791 Hail: INFO: Found 4376 samples in fam file.
2026-08-29 22:51:56.791 Hail: INFO: Found 619950 variants in bim file.
2026-08-29 22:51:57.895 Ha

FatalError: OutOfMemoryError: Java heap space

Java stack trace:
java.lang.OutOfMemoryError: Java heap space
	at 



Hail version: 0.2.133-4c60fddb171a
Error summary: OutOfMemoryError: Java heap space

In [ ]:
pruned_hts = []

# Loop through chromosomes individually to keep memory constant
for chrom in range(1, 23):
    chr_str = f"chr{chrom}" # or str(chrom) depending on your reference
    chr_mt = filtered_mt.filter_rows(filtered_mt.locus.contig == chr_str)
    
    # Checkpoint chromosome sub-table
    chr_mt = chr_mt.checkpoint(f'/vscratch/grp-songyao/pnfioric/temp_dir/tmp_{chr_str}.mt', overwrite=True)
    
    # Prune per chromosome
    chr_pruned_ht = hl.ld_prune(chr_mt.GT, r2=0.1, bp_window_size=250000)
    pruned_hts.append(chr_pruned_ht)

# Combine pruned Table indices across all chromosomes
full_pruned_ht = pruned_hts[0].union(*pruned_hts[1:])

# Step 6: Outputing Dataset as Pruned Data for Admixture

In [ ]:
output_prefix = "/projects/rpci/songyao/pnfioric/Ancestry_Calculation_PW_WCHS/files_for_admixture/wchs_pw_1kgp_hgdp"
print(f"Exporting to single genome-wide PLINK dataset: {output_prefix}")

hl.export_plink(
    final_mt, 
    output_prefix, 
    ind_id=final_mt.s,
    fam_id=final_mt.s
)

print("Hail pipeline completed successfully!")

In [ ]:
def clean_mt(mt):
    # Standardize column schema and genotype entries
    mt = mt.select_cols()
    mt = mt.select_entries(mt.GT)
    
    # Filter variants: autosomal, bi-allelic SNPs only
    mt = mt.filter_rows(mt.locus.in_autosome())
    mt = mt.filter_rows(hl.len(mt.alleles) == 2)
    mt = mt.filter_rows(hl.is_snp(mt.alleles[0], mt.alleles[1]))
    
    return mt

# -------------------------------------------------------------------------
# Step A: Reference Panel (1000G / HGDP)
# -------------------------------------------------------------------------
print("Processing Reference Panel VCFs...")
base_dir = '/projects/rpci/shared/references/1000G_HGDP_v3_gnomAD/VCF/'
chroms = [f'chr{i}' for i in range(1, 23)]
vcf_files = [f'{base_dir}/hgdp_1kg_{chrom}.vcf.bgz' for chrom in chroms]

print("Processing Reference Panel VCFs...")
mt_hgdp_1kg = hl.import_vcf(
    vcf_files, 
    reference_genome='GRCh38', 
    force_bgz=True,
    min_partitions=2000
)
ref_mt = clean_mt(mt_hgdp_1kg)
ref_mt = ref_mt.repartition(1000) # Prevents Java 32-bit array overflow
ref_mt = ref_mt.checkpoint('/vscratch/grp-songyao/pnfioric/temp_dir/ref_hgdp_clean.mt', overwrite=True)

# -------------------------------------------------------------------------
# Step B: Dataset 2 (WCHS PLINK)
# -------------------------------------------------------------------------
print("Processing Dataset 2 (WCHS)...")
dataset2_mt = hl.import_plink(
    bed="/projects/rpci/wchs/pnfioric/WCHS_Merged_0.01_0.3_AABC_AMBER/WCHS_Full_genotypes.bed",
    bim="/projects/rpci/wchs/pnfioric/WCHS_Merged_0.01_0.3_AABC_AMBER/WCHS_Full_genotypes.bim",
    fam="/projects/rpci/wchs/pnfioric/WCHS_Merged_0.01_0.3_AABC_AMBER/WCHS_Full_genotypes.fam",
    reference_genome="GRCh38"
)
dataset2_mt = clean_mt(dataset2_mt)
dataset2_mt = dataset2_mt.repartition(500)
dataset2_mt = dataset2_mt.checkpoint('/vscratch/grp-songyao/pnfioric/temp_dir/dataset2_clean.mt', overwrite=True)

# -------------------------------------------------------------------------
# Step C: Dataset 3 (PATHWAYS PLINK)
# -------------------------------------------------------------------------
print("Processing Dataset 3 (Pathways)...")
plink_mts = [
    hl.import_plink(
        bed=f"{base_path}/pw_TOPMed_chr_{chrom}.bed",
        bim=f"{base_path}/pw_TOPMed_chr_{chrom}.bim",
        fam=f"{base_path}/pw_TOPMed_chr_{chrom}.fam",
        reference_genome="GRCh38"
    ) for chrom in chroms_ds3
]
dataset3_mt = plink_mts[0].union_rows(*plink_mts[1:])
dataset3_mt = clean_mt(dataset3_mt)
dataset3_mt = dataset3_mt.repartition(1000)
dataset3_mt = dataset3_mt.checkpoint('/vscratch/grp-songyao/pnfioric/temp_dir/dataset3_clean.mt', overwrite=True)

Processing Reference Panel VCFs...
Processing Reference Panel VCFs...


2026-09-02 18:54:24.077 Hail: INFO: scanning VCF for sortedness...
2026-09-02 18:56:42.146 Hail: INFO: Coerced sorted VCF - no additional import work to do
